# I-E02 — I-N120 + I-S101 Weighted Soft-Voting

I-N120의 높은 Dice를 유지하면서 I-S101의 미탐 보완 효과를 확인한다.

비교 비중: `N120:S101 = 0.8:0.2, 0.7:0.3, 0.6:0.4, 0.5:0.5`.
각 조합은 동일 38-case validation set에서 threshold sweep을 수행하며, `FP/scan ≤ 1.5`에서 lesion F2 최고 threshold를 선택한다.


In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
LABEL_DIR = Path("/home/stu03/workspace/datasets/ischemic/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations")
N120_PROB = REPO / "runs/I-nnunet/Dataset502_ISLES2022/nnUNetTrainerIschemiaMultimodal__IschemiaPlansDualEncoderGatedV1__3d_fullres/fold_0/best_probability_validation"
S101_PROB = REPO / "runs/I-segresnet/I-S101/final_metrics/val_probabilities"
S101_REF = REPO / "runs/I-segresnet/I-S101/final_metrics/val_predictions"
OUT_ROOT = REPO / "runs/ischemia_ensemble/I-E02_N120_S101_weighted"
WEIGHTS = [(0.8, 0.2), (0.7, 0.3), (0.6, 0.4), (0.5, 0.5)]
assert all(x.exists() for x in [PYTHON, LABEL_DIR, N120_PROB, S101_PROB, S101_REF])
WEIGHTS


## 모든 비중 조합 실행


In [ ]:
for n120_weight, s101_weight in WEIGHTS:
    tag = f"N120_{n120_weight:.1f}_S101_{s101_weight:.1f}"
    output = OUT_ROOT / tag
    command = [
        str(PYTHON), str(REPO / "scripts/ischemia/ensemble/evaluate_soft_voting.py"),
        "--model", f"I-N120={N120_PROB}",
        "--model", f"I-S101={S101_PROB}::{S101_REF}",
        "--weight", str(n120_weight), "--weight", str(s101_weight),
        "--label-dir", str(LABEL_DIR), "--output-dir", str(output),
        "--max-fp-per-scan", "1.5",
    ]
    print(f"[RUN] {tag}")
    subprocess.run(command, cwd=REPO, check=True)


## 조합별 추천 결과


In [ ]:
rows = []
for n120_weight, s101_weight in WEIGHTS:
    tag = f"N120_{n120_weight:.1f}_S101_{s101_weight:.1f}"
    result = json.loads((OUT_ROOT / tag / "ensemble_summary.json").read_text())
    row = {"N120_weight": n120_weight, "S101_weight": s101_weight, **result["selected"]}
    rows.append(row)
comparison = pd.DataFrame(rows).sort_values(["lesion_f2", "macro_dice_positive"], ascending=False)
display(comparison[[
    "N120_weight", "S101_weight", "threshold", "macro_dice_positive",
    "voxel_recall", "voxel_precision", "lesion_recall", "lesion_precision",
    "lesion_f1", "lesion_f2", "small_lesion_recall", "fp_per_scan",
]])
print("Recommended candidate: first row (lesion F2, then Dice; FP/scan constraint already applied).")


## 최종 후보의 threshold 곡선


In [ ]:
best = comparison.iloc[0]
tag = f"N120_{best.N120_weight:.1f}_S101_{best.S101_weight:.1f}"
sweep = pd.read_csv(OUT_ROOT / tag / "ensemble_threshold_sweep.csv")
ensemble = sweep[sweep["candidate"].str.startswith("soft_vote__")]
display(ensemble[[
    "threshold", "macro_dice_positive", "lesion_recall", "lesion_precision",
    "lesion_f2", "small_lesion_recall", "fp_per_scan",
]].sort_values("threshold"))
